# Lap Time Model

Goal: predict lap time from tyre state, fuel load and circuit, then use the model to find the optimal pit-stop lap.

In [30]:
import pandas as pd
import numpy as np

laps = pd.read_parquet("../data/processed/laps_model.parquet")
print(laps.shape)

(78233, 37)


In [31]:
race_driver = ["Year", "Round", "Driver"]

laps["DriverRaceMedian"] = laps.groupby(race_driver)["LapTimeSec"].transform("median")
laps["LapDelta"] = laps["LapTimeSec"] - laps["DriverRaceMedian"]

laps["LapDelta"].describe()

count    78233.000000
mean         0.027794
std          1.122637
min         -8.463000
25%         -0.621500
50%          0.000000
75%          0.650000
max         37.930000
Name: LapDelta, dtype: float64

In [32]:
features = ["TyreLife", "Compound", "LapNumber", "Event", "Year", "Team", "FreshTyre"]
target = "LapDelta"

model_data = laps[features + [target]].copy()

for col in ["Compound", "Event", "Team"]:
    model_data[col] = model_data[col].astype("category")

model_data.dtypes

TyreLife      float64
Compound     category
LapNumber     float64
Event        category
Year            int64
Team         category
FreshTyre        bool
LapDelta      float64
dtype: object

In [33]:
is_train = model_data["Year"] <= 2024
is_test = model_data["Year"] == 2025

X_train, y_train = model_data.loc[is_train, features], model_data.loc[is_train, target]
X_test, y_test = model_data.loc[is_test, features], model_data.loc[is_test, target]

print("train:", X_train.shape, " test:", X_test.shape)

train: (57352, 7)  test: (20881, 7)


In [34]:
from sklearn.metrics import mean_absolute_error

naive_pred = np.zeros(len(y_test))
naive_mae = mean_absolute_error(y_test, naive_pred)
print(f"Naive MAE: {naive_mae:.3f} s")

Naive MAE: 0.796 s


In [35]:
from sklearn.linear_model import LinearRegression

X_all_lin = pd.get_dummies(
    model_data[features],
    columns=["Compound", "Event", "Team"],
    drop_first=True,
    dtype=float,
)
X_train_lin = X_all_lin[is_train]
X_test_lin = X_all_lin[is_test]

lin = LinearRegression()
lin.fit(X_train_lin, y_train)

lin_pred = lin.predict(X_test_lin)
lin_mae = mean_absolute_error(y_test, lin_pred)
print(f"Linear MAE: {lin_mae:.3f} s")

Linear MAE: 0.594 s


In [36]:
from lightgbm import LGBMRegressor

X_train_gb = X_train.astype({"FreshTyre": int})
X_test_gb = X_test.astype({"FreshTyre": int})

gbm = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1,
)
gbm.fit(X_train_gb, y_train)

gbm_pred = gbm.predict(X_test_gb)
gbm_mae = mean_absolute_error(y_test, gbm_pred)
print(f"LightGBM MAE: {gbm_mae:.3f} s")

LightGBM MAE: 0.603 s


In [37]:
results = pd.DataFrame({
    "Model": ["Naive (predict 0)", "Linear regression", "LightGBM"],
    "MAE (s)": [naive_mae, lin_mae, gbm_mae],
})
results["Improvement vs naive"] = (1 - results["MAE (s)"] / naive_mae) * 100
results.round(3)

,Model,MAE (s),Improvement vs naive
0,Naive (predict 0),0.796,0.000
1,Linear regression,0.594,25.365
2,LightGBM,0.603,24.210


In [38]:
lin_train_mae = mean_absolute_error(y_train, lin.predict(X_train_lin))
gbm_train_mae = mean_absolute_error(y_train, gbm.predict(X_train_gb))

print(f"Linear   train: {lin_train_mae:.3f}   test: {lin_mae:.3f}")
print(f"LightGBM train: {gbm_train_mae:.3f}   test: {gbm_mae:.3f}")

Linear   train: 0.587   test: 0.594
LightGBM train: 0.396   test: 0.603


## 5.10 Model parameters

In [39]:
print(gbm.get_params())

coef = pd.Series(lin.coef_, index=X_train_lin.columns)
coef[["TyreLife", "LapNumber", "Compound_MEDIUM", "Compound_SOFT", "FreshTyre"]].round(4)

{'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0, 'importance_type': 'split', 'learning_rate': 0.05, 'max_depth': -1, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 500, 'n_jobs': None, 'num_leaves': 31, 'objective': None, 'random_state': 42, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 1.0, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'verbose': -1}


TyreLife           0.0274
LapNumber         -0.0468
Compound_MEDIUM    0.0758
Compound_SOFT     -0.0006
FreshTyre          0.0631
dtype: float64

In [40]:
is_tr = model_data["Year"] <= 2023
is_val = model_data["Year"] == 2024

X_tr = model_data.loc[is_tr, features].astype({"FreshTyre": int})
y_tr = model_data.loc[is_tr, target]
X_val = model_data.loc[is_val, features].astype({"FreshTyre": int})
y_val = model_data.loc[is_val, target]

print("train:", X_tr.shape, " val:", X_val.shape, " test:", X_test_gb.shape)

train: (36359, 7)  val: (20993, 7)  test: (20881, 7)


In [41]:
import lightgbm as lgb

gbm_reg = LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=200,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1,
)
gbm_reg.fit(
    X_tr, y_tr,
    eval_X=X_val,
    eval_y=y_val,
    eval_metric="l1",
    callbacks=[lgb.early_stopping(100, verbose=False)],
)

best_iter = gbm_reg.best_iteration_
print("best number of trees:", best_iter)

best number of trees: 77


In [42]:
final_gbm = LGBMRegressor(**{**gbm_reg.get_params(), "n_estimators": best_iter})
final_gbm.fit(X_train_gb, y_train)

final_train_mae = mean_absolute_error(y_train, final_gbm.predict(X_train_gb))
final_test_mae = mean_absolute_error(y_test, final_gbm.predict(X_test_gb))
print(f"Regularised LightGBM  train: {final_train_mae:.3f}   test: {final_test_mae:.3f}")

Regularised LightGBM  train: 0.526   test: 0.554


## 5.15 Experiment tracking with MLflow

In [43]:
import mlflow
from pathlib import Path

db_path = Path("../mlflow.db").resolve().as_posix()
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("lap-time-model")

<Experiment: artifact_location=('file:///c:/Users/vova1/Documents/vscodefiles/resume projects/f1 '
 'project/notebooks/mlruns/1'), creation_time=1790736436955, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1790736436955, lifecycle_stage='active', name='lap-time-model', tags={}, trace_location=None, workspace='default'>

In [44]:
def log_run(name, params, train_mae, test_mae):
    with mlflow.start_run(run_name=name):
        mlflow.log_params(params)
        mlflow.log_metric("train_mae", train_mae)
        mlflow.log_metric("test_mae", test_mae)

In [45]:
log_run("linear_regression", {"model": "LinearRegression"}, lin_train_mae, lin_mae)

log_run("lightgbm_default",
        {"model": "LightGBM", "n_estimators": 500, "num_leaves": 31, "min_child_samples": 20},
        gbm_train_mae, gbm_mae)

log_run("lightgbm_regularised",
        {"model": "LightGBM", "n_estimators": best_iter, "num_leaves": 15,
         "min_child_samples": 200, "subsample": 0.8, "colsample_bytree": 0.8, "reg_lambda": 1.0},
        final_train_mae, final_test_mae)

## 5.16 Save and register the final model

In [46]:
import mlflow.lightgbm

with mlflow.start_run(run_name="lightgbm_final"):
    mlflow.log_params(final_gbm.get_params())
    mlflow.log_metric("train_mae", final_train_mae)
    mlflow.log_metric("test_mae", final_test_mae)
    mlflow.lightgbm.log_model(
        final_gbm,
        name="model",
        input_example=X_test_gb.head(5),
        registered_model_name="lap-time-lgbm",
    )

c:\Users\vova1\miniconda3\envs\f1\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/09/29 19:57:11 WARNING mlflow.models.signature: Failed to infer the model signature from the input example. Reason: MlflowException("Failed to enforce schema of data '       TyreLife Compound  LapNumber                  Event  Year

## Q5. On which lap should a driver pit to minimise total race time?

# Pit-Stop Strategy

## 5A. Pit-stop time loss per circuit

### 5A.1 Setup

In [47]:
import fastf1
import pandas as pd
import numpy as np

fastf1.Cache.enable_cache("../cache")
fastf1.set_log_level("ERROR")

laps_model = pd.read_parquet("../data/processed/laps_model.parquet")

### 5A.2 Pit loss for one race

In [48]:
def pit_loss_for_race(year, rnd):
    session = fastf1.get_session(year, rnd, "R")
    session.load(telemetry=False, weather=False, messages=False)
    laps = session.laps.copy()
    laps["Sec"] = laps["LapTime"].dt.total_seconds()

    normal = laps[laps["PitInTime"].isna() & laps["PitOutTime"].isna() & (laps["TrackStatus"] == "1")]
    typical = normal.groupby("Driver")["Sec"].median()

    rows = []
    for _, in_lap in laps[laps["PitInTime"].notna()].iterrows():
        driver = in_lap["Driver"]
        out_lap = laps[(laps["Driver"] == driver) & (laps["LapNumber"] == in_lap["LapNumber"] + 1)]
        if out_lap.empty or driver not in typical:
            continue
        out_lap = out_lap.iloc[0]
        if in_lap["TrackStatus"] != "1" or out_lap["TrackStatus"] != "1":
            continue
        loss = in_lap["Sec"] + out_lap["Sec"] - 2 * typical[driver]
        rows.append({"Year": year, "Round": rnd, "Driver": driver, "PitLoss": loss})

    return pd.DataFrame(rows)

In [49]:
monza = pit_loss_for_race(2024, 16)
print(monza["PitLoss"].describe())
monza.head()

count    30.000000
mean     27.532367
std       4.318844
min      22.648000
25%      24.364500
50%      26.552500
75%      28.423250
max      41.391000
Name: PitLoss, dtype: float64


,Year,Round,Driver,PitLoss
0,2024,16,LEC,27.644
1,2024,16,PIA,27.229
2,2024,16,PIA,24.603
3,2024,16,NOR,26.055
4,2024,16,NOR,24.285


In [50]:
races = (
    laps_model[laps_model["Year"] >= 2024][["Year", "Round", "Event"]]
    .drop_duplicates()
    .sort_values(["Year", "Round"])
)

all_stops = []
for _, race in races.iterrows():
    try:
        stops = pit_loss_for_race(race["Year"], race["Round"])
        stops["Event"] = race["Event"]
        all_stops.append(stops)
        print(f"{race['Year']} R{race['Round']}: {len(stops)} stops")
    except Exception as e:
        print(f"{race['Year']} R{race['Round']}: FAILED - {e}")

pit_stops = pd.concat(all_stops, ignore_index=True)
print(pit_stops.shape)

2024 R1: 38 stops
2024 R2: 5 stops
2024 R3: 31 stops
2024 R4: 35 stops
2024 R5: 22 stops
2024 R6: 12 stops
2024 R7: 27 stops
2024 R8: 5 stops
2024 R9: 19 stops
2024 R10: 42 stops
2024 R11: 44 stops
2024 R12: 45 stops
2024 R13: 40 stops
2024 R14: 32 stops
2024 R15: 26 stops
2024 R16: 30 stops
2024 R17: 17 stops
2024 R18: 23 stops
2024 R19: 21 stops
2024 R20: 21 stops
2024 R22: 39 stops
2024 R23: 8 stops
2024 R24: 22 stops
2025 R1: 0 stops
2025 R2: 24 stops
2025 R3: 21 stops
2025 R4: 28 stops
2025 R5: 16 stops
2025 R6: 9 stops
2025 R7: 12 stops
2025 R8: 36 stops
2025 R9: 40 stops
2025 R10: 28 stops
2025 R11: 30 stops
2025 R12: 16 stops
2025 R13: 26 stops
2025 R14: 29 stops
2025 R15: 10 stops
2025 R16: 19 stops
2025 R17: 15 stops
2025 R18: 22 stops
2025 R19: 18 stops
2025 R20: 26 stops
2025 R21: 33 stops
2025 R22: 16 stops
2025 R23: 23 stops
2025 R24: 27 stops
(1128, 5)


In [51]:
pit_stops = pit_stops[pit_stops["PitLoss"].between(10, 45)]

pit_loss = (
    pit_stops.groupby("Event")["PitLoss"]
    .agg(["median", "count"])
    .rename(columns={"median": "PitLoss", "count": "Stops"})
    .sort_values("PitLoss")
    .reset_index()
)
pit_loss.round(1)

,Event,PitLoss,Stops
0,Monaco Grand Prix,19.1,41
1,Canadian Grand Prix,19.7,45
2,Saudi Arabian Grand Prix,20.2,21
3,Belgian Grand Prix,20.3,45
4,Australian Grand Prix,20.5,30
5,Hungarian Grand Prix,20.5,69
6,Miami Grand Prix,20.6,21
7,Austrian Grand Prix,21.3,74
8,United States Grand Prix,21.8,39
9,Azerbaijan Grand Prix,21.9,32


In [52]:
pit_loss.to_parquet("../data/processed/pit_loss.parquet")